In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score, precision_recall_curve
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import SplineTransformer

OUTPUT_ROOT = Path("../../outputs")
RESULT_DIR = OUTPUT_ROOT / "model_evaluation"
RESULT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_SETS = {"Full": OUTPUT_ROOT / "full_models", "Reduced": OUTPUT_ROOT / "reduced_models"}
COHORTS = {"emer": "Emergency", "elec": "Elective"}
OUTCOMES = {"30D": "30-day mortality", "60D": "60-day mortality", "90D": "90-day mortality", "surgco": "Surgery-related complication", "medco": "Medical-related complication"}
EPSILON = 1e-6

def logit(probability):
    probability = np.clip(np.asarray(probability, float), EPSILON, 1 - EPSILON)
    return np.log(probability / (1 - probability))

def calibration_intercept_slope(y_true, probability):
    x = logit(probability)
    y_true = np.asarray(y_true, float)
    def negative_log_likelihood(parameters):
        linear_predictor = parameters[0] + parameters[1] * x
        return np.sum(np.logaddexp(0, linear_predictor) - y_true * linear_predictor)
    fitted = minimize(negative_log_likelihood, [0.0, 1.0], method="BFGS")
    return float(fitted.x[0]), float(fitted.x[1])

def calibration_errors(y_true, probability):
    x = logit(probability).reshape(-1, 1)
    smoother = make_pipeline(SplineTransformer(n_knots=5, degree=3, include_bias=False), LogisticRegression(C=1e6, max_iter=3000))
    smoother.fit(x, y_true)
    observed = smoother.predict_proba(x)[:, 1]
    error = np.abs(observed - probability)
    return float(error.mean()), float(np.median(error)), float(np.quantile(error, 0.9))

def read_prediction(model_directory, split, final_only=False):
    suffix = "T" if split == "Training" else "V"
    files = sorted(model_directory.glob(f"roc_data_*_{suffix}.csv"))
    if final_only:
        files = [file for file in files if "VotingClassifier" in file.name or "Soft_Blend" in file.name]
    for file in files:
        frame = pd.read_csv(file)
        yield file.stem.replace("roc_data_", "").replace(f"_{suffix}", ""), frame["y_true"].astype(int).to_numpy(), frame["y_score"].astype(float).to_numpy()

rows = []
for model_set, model_root in MODEL_SETS.items():
    for cohort_code, cohort in COHORTS.items():
        for outcome_code, outcome in OUTCOMES.items():
            directory = model_root / f"{outcome_code}_{cohort_code}"
            for dataset in ("Training", "Validation"):
                predictions = list(read_prediction(directory, dataset, final_only=True))
                if len(predictions) != 1:
                    raise ValueError(f"Expected one final blended prediction file in {directory} for {dataset}")
                _, y_true, probability = predictions[0]
                intercept, slope = calibration_intercept_slope(y_true, probability)
                ici, e50, e90 = calibration_errors(y_true, probability)
                rows.append({
                    "Model_set": model_set,
                    "Cohort": cohort,
                    "Outcome": outcome,
                    "Dataset": dataset,
                    "AUROC": roc_auc_score(y_true, probability),
                    "AUPRC": average_precision_score(y_true, probability),
                    "Brier": brier_score_loss(y_true, probability),
                    "Calibration_intercept": intercept,
                    "Calibration_slope": slope,
                    "ICI": ici,
                    "E50": e50,
                    "E90": e90,
                })

results = pd.DataFrame(rows)
results.to_csv(RESULT_DIR / "final_model_performance.csv", index=False)

for cohort_code, cohort in COHORTS.items():
    figure, axes = plt.subplots(len(OUTCOMES), 2, figsize=(13, 22))
    for row_index, (outcome_code, outcome) in enumerate(OUTCOMES.items()):
        directory = MODEL_SETS["Full"] / f"{outcome_code}_{cohort_code}"
        for column_index, dataset in enumerate(("Training", "Validation")):
            axis = axes[row_index, column_index]
            for model, y_true, probability in read_prediction(directory, dataset):
                precision, recall, _ = precision_recall_curve(y_true, probability)
                axis.plot(recall, precision, label=f"{model} ({average_precision_score(y_true, probability):.2f})")
            axis.axhline(y_true.mean(), linestyle="--", color="0.6", label=f"Prevalence ({y_true.mean():.3f})")
            axis.set(xlabel="Recall", ylabel="Precision", title=f"{outcome} - {dataset}", xlim=(0, 1), ylim=(0, 1))
            axis.grid(alpha=0.2)
            axis.legend(fontsize=7, frameon=False)
    figure.suptitle(f"Full models - {cohort} surgery: precision-recall curves")
    figure.tight_layout()
    figure.savefig(RESULT_DIR / f"precision_recall_full_{cohort_code}.png", dpi=300, bbox_inches="tight")
    plt.close(figure)
